## Import Stuff

In [1]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/constants.py"

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3432: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.9/site-packa

In [2]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/FUNCTIONS/load_functions"

fortran mie routines unavailable


/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:47: UserWarning: could not import compiled mie code - mie calculation will be slow
  warnings.warn(


In [3]:
%matplotlib inline

In [4]:
bands        = ["Band 4", "Band 4 nterms2", "Band 4 nterms2 robust -1", "Band 4 nterms2 smooth", "Band 4 nterms2 smooth B6", "Band 4 nterms2 smooth B6 B7", "Band 5", "Band 5 robust -1", "Band 5 robust -2", "Band 6", "Band 6 smooth", "Band 6 smooth B7", "Band 7 nterms2", "Band 7 nterms2 smooth", "Band 7 nterms2 smooth B6"]
bands_naming = ["Band 4", "Band4_nterms2",  "Band4_nterms2_robust_minus",  "Band4_nterms2_smooth",  "Band4_nterms2_smooth_B6",  'Band4_nterms2_smooth_B6_B7',  'Band5',  "Band5_robust_minus1", "Band5_robust_minus2", "Band6",  'Band6_smooth',  'Band6_smooth_B7',  "Band7_nterms2", 'Band7_nterms2_smooth',    'Band7_nterms2_smooth_B6']

lambda_bands_cm = mm_to_cm([lambda_mm[b] for b in bands])

## Set $\lambda$ and $a_{max}$ we want to test

In [5]:
# Set up wavelength array
logwave_vals = np.linspace(0.1, 4, 10000)

lambda_dist_micron = 10**logwave_vals

lambda_dist_cm = micron_to_cm(lambda_dist_micron)

In [6]:
# a_max values for this plot
a_max_test_micron = [1, 100]
a_max_test_cm = micron_to_cm(a_max_test_micron)

In [7]:
f_values = [1, 0.75, 0.5, 0.25, ]#0.3, 0.1, 0.01]

In [10]:
minn = {
    1: 50 / 1,
    0.75: 50 / 0.75,
    0.5: 50 / 0.5, # previously 30 / 0.5,
    0.25: 15 / 0.25, # previously 5 / 0.25,
}

maxx = {
    1: 501 / 1,
    0.75: 501 / 0.75,
    0.5: 501/0.5, # previously 1000 / 0.5,
    0.25: 750/0.25 # previouslt 1000 / 0.25,
}

N = {
    1: (501 - 50) + 1,
    0.75: (501 - 50) + 1,
    0.5: 500,
    0.25: 500,
}


fine_grid = {
    1: 165,
    0.75: 207,
    0.5: 274,
    0.25: 433,
}

In [11]:
# a_min_micron = 10
# a_max_micron = 2e4
# N_grains = 1000

# a_max_log = np.logspace(np.log10(a_min_micron),
#                         np.log10(a_max_micron),
#                         N_grains)

# a_max_fine = np.arange(150, 301, 1)
# # a_max_fine = np.arange(200, 210, 1)

# a_max_dist_micron = np.unique(np.concatenate([a_max_log, a_max_fine]))
# a_max_dist_cm = micron_to_cm(a_max_dist_micron)

n = 5

f_values_data = [0.25]
for f in f_values_data:
    
    # Coarse grid in microns
    a_max_coarse_micron = np.linspace(minn[f],
                                      maxx[f],
                                      int(N[f]))
    
    # Fine grid around best-fit a_max, in microns
    a_max_fine_micron = np.arange(fine_grid[f] - n,
                                  fine_grid[f] + n + 1,
                                  1)
    
    # Combine, remove duplicates, sort
    a_max_dist_micron = np.unique(np.concatenate([
        a_max_coarse_micron,
        a_max_fine_micron
    ]))
    
    # Optional safety: only keep values inside your bounds
    a_max_dist_micron = a_max_dist_micron[
        (a_max_dist_micron >= minn[f]) &
        (a_max_dist_micron <= maxx[f])
    ]
    
    # Convert to cm for DSHARP
    a_max_dist_cm = micron_to_cm(a_max_dist_micron)
    
    print(rf'f = {f}')
    print(len(a_max_coarse_micron))
    print(len(a_max_fine_micron))
    print(len(a_max_dist_micron))
    print(' ')
    
    # Run DSHARP for this f
    P, omega, P_times_omega = run_DSHARP(f, a_max_test_cm, a_max_dist_cm, lambda_bands_cm, lambda_dist_cm)
    
    # Prepare data dictionary
    data = {
        "a_max_micron": cm_to_micron(a_max_dist_cm)
    }
    
    # Add per-band columns
    for i, b in enumerate(bands_naming):  # bands = [4, 5, 6, ...]
        data[f"P_{b}"] = P[:, i]
        data[f"omega_{b}"] = omega[:, i]
        data[f"P_times_omega_{b}"] = P_times_omega[:, i]
    
    # Convert to DataFrame
    df = pd.DataFrame(data)
    
    # Save to CSV with f in the filename
    f_str = str(f).replace('.', '_')  # e.g., 0.3 → '0_3' for filename
    file_name = f"P_omega_vs_amax_{f_str}_JUST_AMAX_POSTER.csv"
    df.to_csv(P_omega_data_folder_path + 'Poster/'+ file_name, index=False)
    
    print(f"Saved {file_name}")

f = 0.25
500
11
511
 
Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
using Maxwell-Garnett mixing: first component should be host material (= matrix)
    matrix = Vacuum
Mie ... Done!
Mie ... Done!
Saved P_omega_vs_amax_0_25_JUST_AMAX_POSTER.csv
